# Metabolic Engineering and Flux Balance Analysis
In this lecture, we will use flux balance analysis (FBA) to estimate reaction rates in a metabolic network. We will construct the stoichiometric matrix, formulate the flux estimation problem as a linear program, and use reaction reversibility and enzyme capacity to set the flux bounds. In the example, we apply these ideas to calculate the maximum urea production rate in a small reaction network.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Represent a metabolic reaction network:__ Construct a stoichiometric matrix from biochemical reactions and interpret its rows, columns, and coefficient signs.
> * __Formulate a flux balance analysis problem:__ Combine steady-state metabolite balances, flux bounds, and a linear objective, and explain the role of each in determining an optimal flux distribution.
> * __Develop and interpret flux bounds:__ Explain how reaction reversibility, enzyme abundance, allosteric activity, and substrate saturation enter the bounds model, and state the assumptions behind its simplified form.

Let's get started!

___

## Examples
We will use the following example to connect the flux balance formulation to a numerical calculation:

> [▶ Calculate fluxes in a urea-cycle model](CHEME-5800-L6a-Example-UreaCycle-FluxBalance-Fall-2026.ipynb). In this example, we will build a stoichiometric model of the urea cycle, estimate reaction directions and enzyme capacities, and solve a linear program to maximize urea export. We will use the resulting fluxes and material balances to identify the reaction capacity that limits production.

___


## Metabolic Engineering
Metabolic pathways share metabolites, so changing one reaction can affect the supply of material to other reactions. To increase production of a desired compound, we need to consider how material flows through the network. We can explore these connections in the [KEGG metabolic pathways map (map01100)](https://www.kegg.jp/pathway/map01100). The simplified map below highlights selected connections in central metabolism.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<img class="course-diagram" src="figs/Fig-Central-Metabolism/Fig-Central-Metabolism.svg" width="1100" alt="Selected routes through glycolysis, the pentose phosphate pathway, and the TCA cycle, with branches toward lactate, nucleotides, lipids, and proteins. ATP and redox-carrier yields accompany the pathways, with a circular TCA cycle and an oxidative phosphorylation inset. Arrows may combine multiple reactions.">

Several metabolites in this map feed more than one route: glucose 6-phosphate enters glycolysis or the pentose phosphate pathway, pyruvate becomes lactate or acetyl-CoA, and acetyl-CoA enters the TCA cycle or lipid synthesis.

> __What is metabolic engineering?__
>
> Metabolic engineering is the directed modification of an organism's metabolic pathways through genetic and regulatory changes to increase the production of a desired compound or to enable the synthesis of new products. The engineering question is: how can we redirect the flow of carbon, nitrogen, and energy through the network toward a target molecule while meeting the cell's other requirements?
> 
> Two landmark papers that defined the field appeared in the same 1991 issue of _Science_:
> * [Bailey JE. Toward a science of metabolic engineering. Science. 1991 Jun 21;252(5013):1668–75.](https://pubmed.ncbi.nlm.nih.gov/2047876/)
> * [Stephanopoulos G, Vallino JJ. Network rigidity and metabolic engineering in metabolite overproduction. Science. 1991 Jun 21;252(5013):1675–81.](https://pubmed.ncbi.nlm.nih.gov/1904627/)

We describe the flow of material through the network using __metabolic fluxes__: reaction rates expressed per unit volume or biomass. Which combinations of reaction rates can sustain production of a desired compound, and what limits the production rate? Flux balance analysis provides a way to answer these questions.

___

## Flux Balance Analysis
__Flux balance analysis (FBA)__ estimates metabolic reaction rates from material balances and biological constraints, without requiring a kinetic rate law for every reaction. At steady state, each modeled metabolite must be produced or imported as fast as it is consumed or exported. These balances couple the reaction rates but generally allow many combinations, called __flux distributions__.

We restrict the rates using bounds on nutrient uptake, reaction direction, and enzyme capacity, then select an __objective__, such as maximizing urea export. The optimal flux distribution shows what the network can achieve under those constraints; it does not establish how the cell actually operates.

### Metabolic Networks and Data Sources
Writing the balances requires knowing which reactions consume or produce each metabolite and in what amounts. A __metabolic reconstruction__ provides this information through a curated set of reactions, metabolites, and stoichiometric coefficients. It defines the network, not the reaction rates.

The network includes nutrient breakdown (__catabolism__) and synthesis of cellular components and products (__anabolism__). Its scope depends on the organism and the modeling question. When compartments are modeled explicitly, the same compound in the cytosol and mitochondrion is represented as two distinct species connected by transport reactions.

We can obtain reaction networks and supporting biological data from several resources. The linked papers describe what each database provides:

| Resource | What we use it for |
|:---|:---|
| [KEGG (Kanehisa et al., 2025)](https://academic.oup.com/nar/article/53/D1/D672/7824602) | Explore pathways and relationships among genes, enzymes, and reactions. |
| [BioCyc (Karp et al., 2019)](https://pubmed.ncbi.nlm.nih.gov/29447345/) | Examine organism-specific pathway and genome information. |
| [BiGG Models (Norsigian et al., 2020)](https://academic.oup.com/nar/article/48/D1/D402/5614178) | Obtain genome-scale metabolic models with standardized metabolite and reaction identifiers. |
| [BRENDA (Chang et al., 2021)](https://academic.oup.com/nar/article/49/D1/D498/5992283) | Find enzyme properties and kinetic measurements that can inform flux bounds. |
| [BioNumbers (Milo et al., 2010)](https://academic.oup.com/nar/article/38/suppl_1/D750/3112244) | Find measured biological quantities and their literature sources. |

The reaction records give us the stoichiometry; kinetic and physiological measurements help us set the flux bounds. Let's start by organizing the reaction records into a matrix.

### Stoichiometric Matrix
The stoichiometric matrix records the net amount of each species produced or consumed by each reaction. A column describes one reaction; a row collects the contributions of all reactions to one species balance. The same construction works for a whole cell, a single compartment, or a cell-free reaction in a test tube.

> __Definition: Stoichiometric matrix__
>
> Let $\mathcal{M}$ denote the set of modeled chemical species and $\mathcal{R}$ the set of reactions. The stoichiometric matrix is given by:
>
> $$
> \mathbf{S}=[\sigma_{ij}]
> \in\mathbb{R}^{|\mathcal{M}|\times|\mathcal{R}|}.
> $$
>
> Here, $|\mathcal{M}|$ is the number of modeled species and $|\mathcal{R}|$ is the number of reactions. The entry $\sigma_{ij}$ is the net stoichiometric coefficient of species $i$ in reaction $j$, with signs defined relative to the reaction's written forward direction:
>
> * $\sigma_{ij}>0$: the reaction produces species $i$.
> * $\sigma_{ij}<0$: the reaction consumes species $i$.
> * $\sigma_{ij}=0$: the reaction causes no net change in species $i$.
>
> A zero entry can mean that the species is absent from the reaction or that its reactant and product coefficients cancel. The matrix records net stoichiometry; it does not encode every catalytic or regulatory interaction.

For example, consider a reaction $j$ that consumes one unit of $A$ and two units of $B$ to produce one unit of $C$. With the species ordered as $(A,B,C)$, its column is given by:

$$
A+2B\longrightarrow C,
\qquad
\mathbf{s}_j=
\begin{bmatrix}
-1\\
-2\\
1
\end{bmatrix}.
$$

Multiplying the stoichiometric column by the reaction flux gives the reaction's contribution to the three species balances. The coefficient $-2$ means that the consumption rate of $B$ is twice the reaction flux. For a reversible reaction with a negative flux, the signs of these contributions reverse; we keep the same stoichiometric column.

How does material enter or leave the network? An __exchange reaction__ connects a metabolite to the surroundings, written as $\emptyset$, which we do not balance. Its column has a single nonzero entry. For the product $C$, exchange reaction $k$ and its column are given by:

$$
\emptyset\longrightarrow C,
\qquad
\mathbf{s}_k=
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}.
$$

A positive flux brings $C$ in, and a negative flux removes it. The urea example writes every exchange this way; the BiGG models in [L6b](../L6b/CHEME-5800-L6b-Lab-OverflowMetabolism-Fall-2026.ipynb) write $C\longrightarrow\emptyset$, so uptake is negative there.

Adding the contributions from all columns, exchanges included, gives the net production rate of each metabolite. With the fluxes collected in the vector $\hat{\mathbf{v}}$, these rates are the entries of $\mathbf{S}\hat{\mathbf{v}}$, and at steady state we set them to zero:

$$
\mathbf{S}\hat{\mathbf{v}}=\mathbf{0}.
$$

Every modeled metabolite is balanced, yet material crosses the boundary through the exchange fluxes.

### Flux Balance Analysis Formulation
We now express the flux estimation problem as a linear program: the reaction fluxes are the unknowns, the balances and bounds are the constraints, and a weighted sum of fluxes is the objective. The problem has the same structure as the minimum-cost flow problem in [L5c](../../week-05/L5c/CHEME-5800-L5c-Lecture-LinearProgramming-Fall-2026.ipynb): $\mathbf{S}$ plays the role of the incidence matrix $\mathbf{A}$, with the same signs, and the exchange reactions take over the source and sink terms, so the right-hand side is zero. Unlike a column of the incidence matrix, a reaction column can have more than two nonzero entries and coefficients other than $\pm1$.

The balances rest on two assumptions, which the [companion derivation](CHEME-5800-L6a-Advanced-Derivation-FluxBalanceAnalysis-Fall-2026.ipynb) develops from a mole balance. The amount of each metabolite per gram of biomass stays constant, and the extra production needed to offset dilution by new biomass, called growth dilution, is small compared with each metabolite's production and consumption rates. Under these assumptions, the problem is given by:

> __Flux balance analysis as a linear program__
>
> As in the stoichiometric-matrix definition, $\mathcal{M}$ is the set of modeled species and $\mathcal{R}$ is the set of reactions, including exchanges. The coefficient $\sigma_{ij}$, the $(i,j)$ entry of $\mathbf{S}$, is the net stoichiometric coefficient of species $i\in\mathcal{M}$ in reaction $j\in\mathcal{R}$.
>
> For each reaction $j$, $\hat v_j$ is the unknown flux, $c_j$ is its weight in the objective, and $\mathcal{L}_j$ and $\mathcal{U}_j$ are its lower and upper bounds. The vector $\hat{\mathbf{v}}$ collects the fluxes. Fluxes and bounds are expressed in $\mathrm{mmol\,gDW^{-1}\,h^{-1}}$, millimoles per gram of cell dry weight per hour. We fix the stoichiometric coefficients, objective weights, and flux bounds before solving the following problem:
>
> $$
> \begin{aligned}
> \underset{\hat{\mathbf{v}}}{\operatorname{maximize}}\quad
> & \sum_{j\in\mathcal{R}}c_j\hat v_j \\
> \text{subject to}\quad
> & \sum_{j\in\mathcal{R}}\sigma_{ij}\hat v_j=0,
> && i\in\mathcal{M},\\
> & \mathcal{L}_j\leq\hat v_j\leq\mathcal{U}_j,
> && j\in\mathcal{R}.
> \end{aligned}
> $$
>
> In matrix form, the balance constraints are $\mathbf{S}\hat{\mathbf{v}}=\mathbf{0}$ and the objective is $\mathbf{c}^{\top}\hat{\mathbf{v}}$, where $\mathbf{c}$ collects the coefficients $c_j$. A feasible flux distribution satisfies the balances and bounds. The objective selects an optimal distribution, which need not be unique.

In the urea example, urea leaves through an exchange with a negative flux, so we maximize urea export by setting its objective coefficient to $-1$ and all other coefficients to zero. For more on the formulation, see [Orth et al. (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3108565/) and [Heirendt et al. (2019)](https://pubmed.ncbi.nlm.nih.gov/30787451/). Next, we will use enzyme capacity and reaction direction to specify the flux bounds.

___

## A Model for Flux Bounds
The material balances require reaction rates to be consistent with one another, but they do not specify how fast an enzyme can operate or which reaction directions are allowed. Flux bounds supply these restrictions. For an enzyme-catalyzed reaction, we model the available capacity using enzyme abundance, catalytic activity, regulation, and substrate saturation. We use a reversibility parameter to determine whether the reaction can also carry flux in the reverse direction. The resulting bounds are given by:

$$
\underbrace{
-\delta_j\overbrace{
\left[V_{max,j}^{\circ}\left(\frac{e}{e^{\circ}}\right)\theta_j(\dots)f_j(\dots)\right]
}^{\text{assumed reverse capacity}}
}_{\mathcal{L}_j}
\leq\hat v_j\leq
\underbrace{
V_{max,j}^{\circ}\left(\frac{e}{e^{\circ}}\right)\theta_j(\dots)f_j(\dots)
}_{\mathcal{U}_j}.
$$

The maximum reaction velocity at a characteristic enzyme abundance is given by:

$$
V_{max,j}^{\circ}=k_{cat,j}^{\circ}e^{\circ}.
$$

The following table defines the model quantities. We express enzyme abundance and reaction flux per gram of cell dry weight:

<table style="width:100%;table-layout:fixed;text-align:left;">
<colgroup><col style="width:23%;"><col style="width:77%;"></colgroup>
<thead><tr><th style="text-align:left;">Quantity</th><th style="text-align:left;">Meaning</th></tr></thead>
<tbody>
<tr><td><i>V</i><sub>max,j</sub><sup>∘</sup></td><td>Maximum reaction velocity at the characteristic enzyme abundance (mmol gDW<sup>−1</sup> h<sup>−1</sup>).</td></tr>
<tr><td><i>k</i><sub>cat,j</sub><sup>∘</sup></td><td>Reference catalytic turnover number (h<sup>−1</sup>).</td></tr>
<tr><td><i>e</i><sup>∘</sup> and <i>e</i></td><td>Characteristic and actual abundances of the enzyme that catalyzes reaction <i>j</i> (mmol gDW<sup>−1</sup>).</td></tr>
<tr><td><i>θ</i><sub>j</sub>(…) ∈ [0, 1]</td><td>Fraction of maximal enzyme activity that remains under allosteric regulation; (…) stands for the concentrations of the enzyme's activators and inhibitors.</td></tr>
<tr><td><i>f</i><sub>j</sub>(…) ∈ [0, 1]</td><td>Substrate saturation; (…) stands for the substrate concentrations. Equal to one when the substrates are saturating.</td></tr>
<tr><td><i>δ</i><sub>j</sub> ∈ {0, 1}</td><td>Zero for a forward-only reaction; one for a reversible reaction.</td></tr>
</tbody>
</table>

The bounds model assumes equal forward and reverse capacities for reversible reactions; different capacities would need separate parameters for the reverse direction. We fix all of these quantities before solving, so each bound is a number and the problem remains a linear program.

### Simplified Bounds Model
Let's initially assume that the actual enzyme abundance equals its characteristic value, that allosteric regulation does not reduce enzyme activity, and that the substrates are saturating. Setting $e/e^{\circ}=\theta_j=f_j=1$ gives the bounds:

$$
\underbrace{-\delta_jV_{max,j}^{\circ}}_{\mathcal{L}_j}\leq\hat v_j\leq\underbrace{V_{max,j}^{\circ}}_{\mathcal{U}_j}.
$$

To set these simplified bounds for each enzyme-catalyzed reaction, we need estimates of the turnover number, characteristic enzyme abundance, and reversibility parameter. Exchange reactions have separate bounds on uptake and secretion.

> __Example:__
>
> [▶ Estimate fluxes in the urea cycle](CHEME-5800-L6a-Example-UreaCycle-FluxBalance-Fall-2026.ipynb). In this example, we will estimate reaction directions from thermodynamic data and calculate flux bounds from enzyme turnover numbers and an assumed enzyme abundance. We will then maximize urea export and use the material balances to explain which enzyme capacity limits production and how a competing reaction changes the result.

Not every bound affects the optimum. In the example, we use the material balances to find the enzyme capacity that limits urea export.

___

## Summary
We formulated flux balance analysis as a linear program, combining a metabolic reaction network, biological constraints, and a production objective.

> __Key Takeaways:__
>
> * __Stoichiometric matrix:__ We represented each reaction by its net production and consumption of metabolites, and used exchange reactions to let material cross the system boundary. Organizing these coefficients into a matrix allowed us to write the steady-state balances as a single matrix equation.
>
> * __Flux balance analysis:__ We combined steady-state balances, flux bounds, and a linear objective to estimate an optimal flux distribution. The prediction depends on the selected constraints and objective, and the optimal distribution need not be unique.
>
> * __Flux bounds:__ We used enzyme abundance, catalytic activity, regulation, substrate saturation, and reaction reversibility to constrain the allowed rates. Assuming characteristic enzyme abundance, no allosteric effects, and saturating substrates gave us a simpler bounds model.

The same reaction network can support different production rates under different constraints. An optimal flux distribution tells us what the model allows under the conditions we specify.

___